# Lab 04 guiado · Minería de datos

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 4, ruta guiada*

Este cuaderno se abre directo desde GitHub. Lo que escribas se pierde al cerrar la pestaña. Si quieres conservarlo, descárgalo con Archivo, Descargar, Descargar .ipynb.

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#31708f"></td><td bgcolor="#d9edf7"><font color="#0b3d62" size="3"><b>Qué vamos a responder</b><br>1. Cómo se convierte una bitácora de eventos en canastas que se pueden comparar.<br>2. Cómo se mide si dos eventos aparecen juntos más de lo normal.<br>3. Qué evento suele venir después de otro, y en cuántos días.</font></td></tr></table>

---
<br>

## Paso 1 de 9 · ¿Qué hay en la bitácora?

**Qué vamos a responder.** Cuántos eventos de mantenimiento se registraron en el año y cómo se ven.

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 1
DATOS = ("https://raw.githubusercontent.com/RodrigoMSB/"
         "curso-python-minergia/main/datos/")
import pandas as pd

# La bitácora del año, un evento por fila, como la llena el turno.
mant = pd.read_csv(DATOS + "mantenimiento.csv", parse_dates=["fecha"])

print("Eventos del año", len(mant))
print("Centrales con eventos", mant["central"].nunique())
mant[["central", "fecha", "tipo_evento", "duracion_horas"]].head()

---
<br>

## Paso 2 de 9 · ¿Cuánto dura cada tipo de evento?

**Qué vamos a responder.** Qué eventos son más frecuentes y cuáles dejan más horas detenida una central.

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 2
resumen = mant.groupby("tipo_evento")["duracion_horas"].agg(
    eventos="count",
    horas_promedio="mean",
    horas_maximo="max",
)
resumen.round(1).sort_values("eventos", ascending=False)

---
<br>

## Paso 3 de 9 · Tu turno, mira qué le pasa a cada tecnología

**Qué vamos a responder.** Si un tipo de evento se concentra en alguna tecnología.

**Qué hacer.** Elige un tipo de evento en la lista y vuelve a correr la celda. Prueba con evento_climatico y con falla_electrica.

In [ ]:
# Paso 3
tipo = "evento_climatico" #@param ["preventivo", "correctivo", "inspeccion", "falla_mecanica", "falla_electrica", "evento_climatico"]

# A cada evento se le pega la ficha de su central, como en el Lab 01.
centrales = pd.read_csv(DATOS + "centrales.csv")
con_ficha = pd.merge(mant, centrales, on="central", how="left")

elegidos = con_ficha[con_ficha["tipo_evento"] == tipo]
print(f"✓ Hay {len(elegidos)} eventos de tipo {tipo}")
elegidos["tecnologia"].value_counts()

---
<br>

## Paso 4 de 9 · ¿Qué le pasó a cada central en cada mes?

**Qué vamos a responder.** Cómo se agrupan los eventos para compararlos, igual que las compras de un supermercado se agrupan en boletas.

**Qué hacer.** Corre la celda. Cada fila es una canasta, todo lo que le pasó a una central en un mes.

In [ ]:
# Paso 4
mant["mes"] = mant["fecha"].dt.month

# Se juntan los eventos de cada central y mes. set quita los repetidos
# y sorted los ordena por nombre.
canastas = mant.groupby(["central", "mes"])["tipo_evento"].apply(lambda s: sorted(set(s)))

print("✓ Canastas", len(canastas))
print("Eventos distintos por canasta, en promedio", round(canastas.apply(len).mean(), 1))
canastas.head(6).to_frame("lo_que_pasó")

---
<br>

## Paso 5 de 9 · ¿En cuántas canastas aparece cada evento?

**Qué vamos a responder.** Qué tan común es cada tipo de evento, contando canastas y no eventos.

**Qué hacer.** Corre la celda. La tabla de abajo tiene una fila por canasta y una columna por tipo de evento, con un ✓ donde el evento ocurrió.

In [ ]:
# Paso 5
# crosstab cuenta los eventos de cada canasta y > 0 lo convierte en sí o no.
tabla = pd.crosstab([mant["central"], mant["mes"]], mant["tipo_evento"]) > 0
print("Canastas", tabla.shape[0], " tipos de evento", tabla.shape[1])

# El promedio de una columna de sí o no es la proporción de sí.
print()
print("Porcentaje de canastas donde aparece cada tipo")
print((100 * tabla.mean()).round(1).sort_values(ascending=False))

tabla.head(6).map(lambda v: "✓" if v else "")

---
<br>

## Paso 6 de 9 · Tu turno, si pasa una cosa, ¿pasa la otra?

**Qué vamos a responder.** Si dos eventos aparecen juntos en la misma canasta más de lo normal.

**Qué hacer.** Elige un evento en si y otro en entonces, y vuelve a correr la celda. Prueba falla_electrica con correctivo y después falla_electrica con preventivo.

In [ ]:
# Paso 6
si = "falla_electrica" #@param ["preventivo", "correctivo", "inspeccion", "falla_mecanica", "falla_electrica", "evento_climatico"]
entonces = "correctivo" #@param ["preventivo", "correctivo", "inspeccion", "falla_mecanica", "falla_electrica", "evento_climatico"]

if si == entonces:
    print("Elige dos eventos distintos")
else:
    con_si = tabla[si].sum()
    con_los_dos = (tabla[si] & tabla[entonces]).sum()

    # Confianza, de las canastas con si, cuántas tienen también entonces.
    confianza = 100 * con_los_dos / con_si
    # Lo normal, en cuántas canastas aparece entonces, haya o no haya si.
    normal = 100 * tabla[entonces].mean()
    # Lift, cuántas veces más de lo normal. Sobre 1 van juntos, cerca de 1 no hay relación.
    lift = confianza / normal

    print(f"Canastas con {si}", con_si)
    print(f"De ellas, también con {entonces}", con_los_dos)
    print(f"Confianza  {confianza:.1f} por ciento")
    print(f"{entonces} aparece en general en el {normal:.1f} por ciento")
    print(f"Lift  {lift:.2f}")

---
<br>

## Paso 7 de 9 · ¿Qué parejas de eventos van juntas?

**Qué vamos a responder.** Todas las combinaciones a la vez, para no tener que probarlas de a una.

**Qué hacer.** Corre la celda. Están ordenadas por confianza. Mira la columna lift antes de sacar conclusiones.

In [ ]:
# Paso 7
# El mismo cálculo del Paso 6 para todas las parejas posibles.
filas = []
for si in tabla.columns:
    for entonces in tabla.columns:
        if si != entonces:
            juntos = (tabla[si] & tabla[entonces]).sum()
            confianza = juntos / tabla[si].sum()
            filas.append({"si": si, "entonces": entonces, "canastas": juntos,
                          "confianza": round(100 * confianza, 1),
                          "lift": round(confianza / tabla[entonces].mean(), 2)})

reglas = pd.DataFrame(filas)
print("✓ Parejas revisadas", len(reglas))
reglas.sort_values("confianza", ascending=False).head(6)

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>Una confianza alta no basta</b><br>Falla eléctrica con preventivo tiene un 52,4 por ciento de confianza y parece una regla.<br>Pero el preventivo aparece en el 56,3 por ciento de todas las canastas.<br>Su lift es 0,93, o sea que no aparece más de lo normal.<br>Las dos reglas que sí dicen algo tienen lift cercano a 2.<br>Que dos eventos vayan juntos no prueba que uno cause el otro.</font></td></tr></table>

---
<br>

## Paso 8 de 9 · Tu turno, ¿qué viene después de cada evento?

**Qué vamos a responder.** Cuál es el evento siguiente en la misma central, ahora mirando el orden en el tiempo.

**Qué hacer.** Elige un evento en la lista y vuelve a correr la celda. Prueba falla_electrica, evento_climatico y preventivo.

In [ ]:
# Paso 8
evento = "falla_electrica" #@param ["preventivo", "correctivo", "inspeccion", "falla_mecanica", "falla_electrica", "evento_climatico"]

# Se ordena por central y fecha, y shift(-1) trae el evento de la fila siguiente.
ordenados = mant.sort_values(["central", "fecha"]).copy()
ordenados["siguiente"] = ordenados.groupby("central")["tipo_evento"].shift(-1)

# normalize="index" deja cada fila en proporciones que suman uno.
transicion = pd.crosstab(ordenados["tipo_evento"], ordenados["siguiente"],
                         normalize="index")

print(f"Después de {evento}, el evento siguiente en la misma central, en porcentaje")
(100 * transicion.loc[evento]).round(1).sort_values(ascending=False)

---
<br>

## Paso 9 de 9 · Tu turno, ¿en cuántos días llega el correctivo?

**Qué vamos a responder.** Qué porcentaje de las fallas eléctricas tiene un correctivo en la misma central dentro de unos días.

**Qué hacer.** Corre la celda. Después mueve el deslizador a 0, a 1 y a 7, y vuelve a correrla cada vez.

In [ ]:
# Paso 9
dias = 3 #@param {type:"slider", min:0, max:7, step:1}

electricas = mant[mant["tipo_evento"] == "falla_electrica"]
correctivos = mant[mant["tipo_evento"] == "correctivo"]

# Cada falla eléctrica se junta con todos los correctivos de su misma central.
pares = pd.merge(electricas[["central", "fecha"]], correctivos[["central", "fecha"]],
                 on="central", suffixes=("_falla", "_correctivo"))
distancia = (pares["fecha_correctivo"] - pares["fecha_falla"]).dt.days

# Se quedan los correctivos que llegan el mismo día o hasta dias después.
cerca = pares[(distancia >= 0) & (distancia <= dias)]
con_correctivo = cerca[["central", "fecha_falla"]].drop_duplicates()
porcentaje = 100 * len(con_correctivo) / len(electricas)

print("Fallas eléctricas", len(electricas))
print(f"Con un correctivo dentro de {dias} días", len(con_correctivo))
print(f"Medido {porcentaje:.1f} por ciento")
print("Lo que se plantó al crear los datos, 70 por ciento")

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>La ventana la decide quien analiza</b><br>Con 0 días da 14,9 por ciento y con 3 días da 72,3.<br>Desde los 3 días ya no sube más, porque así se crearon los datos.<br>En un informe se escribe la ventana que se usó, porque cambia el resultado.<br>Estos datos son inventados y el patrón se plantó a propósito para poder medirlo.</font></td></tr></table>

---
<br>

## Lo que lograste hoy

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#28a745"></td><td bgcolor="#d8f0df"><font color="#14532d" size="3">• Una bitácora de eventos se puede agrupar en canastas, como boletas de supermercado.<br>• Una tabla de sí o no por canasta permite contar qué tan común es cada evento.<br>• La confianza dice qué tan seguido van juntos y el lift si es más de lo normal.<br>• Una confianza alta con lift cercano a 1 no dice nada.<br>• El orden en el tiempo muestra qué evento suele venir después de otro.<br>• La ventana de días es una decisión que cambia el resultado y se deja escrita.</font></td></tr></table>

## Para seguir practicando

Estos ejercicios son para quien quiera seguir después de clase. Aquí sí se escribe código.
Antes de empezar, corre todo el cuaderno con Entorno de ejecución, Ejecutar todo.
Escribe tu respuesta en la celda vacía de cada ejercicio y abre la solución solo después de intentarlo.

### Ejercicio 1 · Canastas por trimestre

Arma la tabla de sí o no del Paso 5 con canastas por central y trimestre en vez de central y mes. Cuenta cuántas canastas quedan y calcula el porcentaje de canastas donde aparece cada evento. Antes de correr, piensa si los porcentajes van a subir o a bajar. Pista, dt.quarter en vez de dt.month.

In [ ]:
# Ejercicio 1

<details>
<summary>Solución · Ejercicio 1</summary>

```python
mant["trimestre"] = mant["fecha"].dt.quarter
por_trimestre = pd.crosstab([mant["central"], mant["trimestre"]], mant["tipo_evento"]) > 0

print("Canastas por trimestre", len(por_trimestre))
(100 * por_trimestre.mean()).round(1).sort_values(ascending=False)
```

Quedan 80 canastas y todos los porcentajes suben, el preventivo llega al 90,0. Con canastas más grandes casi todo aparece con todo y las reglas pierden fuerza.

</details>

### Ejercicio 2 · Lo que anuncia un correctivo

Al que programa las cuadrillas le sirve saber qué eventos anuncian un correctivo. Filtra la tabla reglas del Paso 7 para quedarte con las que tienen correctivo en entonces y ordénalas por lift de mayor a menor.

In [ ]:
# Ejercicio 2

<details>
<summary>Solución · Ejercicio 2</summary>

```python
reglas[reglas["entonces"] == "correctivo"].sort_values("lift", ascending=False)
```

Solo la falla eléctrica anuncia un correctivo, con lift 2,08. Las demás quedan bajo 1.

</details>

### Ejercicio 3 · Clima y fallas mecánicas el mismo día

Al crear los datos se plantó un segundo patrón, la mitad de los eventos climáticos trae una falla mecánica en la misma central el mismo día. Mídelo con un merge por central y fecha, igual que en el Paso 9 pero sin ventana.

In [ ]:
# Ejercicio 3

<details>
<summary>Solución · Ejercicio 3</summary>

```python
climaticos = mant[mant["tipo_evento"] == "evento_climatico"]
mecanicas = mant[mant["tipo_evento"] == "falla_mecanica"]

dias_con_mecanica = mecanicas[["central", "fecha"]].drop_duplicates()
con_mecanica = pd.merge(climaticos, dias_con_mecanica, on=["central", "fecha"])

print("Eventos climáticos", len(climaticos))
print("Con falla mecánica el mismo día", len(con_mecanica))
print(f"Medido {100 * len(con_mecanica) / len(climaticos):.1f} por ciento, plantado 50")
```

Son 43 de 80, un 53,8 por ciento, cerca del 50 que se plantó.

</details>

### Ejercicio 4 · ¿Qué central se parece a cuál?

Arma el día promedio de cada central, 24 números con su generación media de cada hora, y busca para cada central la más parecida. Cuenta en cuántas de las 20 la más parecida es de la misma tecnología. Hazlo dos veces, con los valores tal cual y dividiendo cada perfil por su máximo para comparar solo la forma. Pista, pivot_table con hora en las columnas y pdist de scipy para las distancias.

In [ ]:
# Ejercicio 4

<details>
<summary>Solución · Ejercicio 4</summary>

```python
from scipy.spatial.distance import pdist, squareform
import numpy as np

gen = pd.read_csv(DATOS + "generacion.csv")
perfil = gen.pivot_table(index="central", columns="hora", values="mwh", aggfunc="mean")
forma = perfil.div(perfil.max(axis=1), axis=0)
tec = centrales.set_index("central")["tecnologia"]

for nombre, datos in [("tamaño y forma", perfil), ("solo la forma", forma)]:
    distancias = pd.DataFrame(squareform(pdist(datos)), index=datos.index, columns=datos.index)
    distancias = distancias.mask(np.eye(len(distancias), dtype=bool))
    vecina = distancias.idxmin(axis=1)
    aciertos = (tec[vecina.index].values == tec[vecina.values].values).sum()
    print(f"Comparando {nombre}, acierta la tecnología en {aciertos} de 20")
```

Con los valores tal cual acierta en 7 de 20, porque una central grande y una chica de la misma tecnología quedan lejos. Comparando solo la forma acierta en 15.

</details>

Las distancias entre perfiles, los algoritmos apriori y FP-Growth y las reglas con más de un evento están en [la ruta completa de este módulo](https://colab.research.google.com/github/RodrigoMSB/curso-python-minergia/blob/main/modulos/modulo04/lab.ipynb).